In [ ]:
import os
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "AIDOCell")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set environmental variables so models are downloaded to the the model path rather than ~/.cache/huggingface
os.environ['HF_HOME'] = MODEL_PATH                        # Primary control
os.environ['HUGGINGFACE_HUB_CACHE'] = MODEL_PATH          # Hub downloads
os.environ['TRANSFORMERS_CACHE'] = MODEL_PATH             # Transformers-specific

from modelgenerator.backbones import aido_cell_3m, aido_cell_10m, aido_cell_100m

# local .py files
from AIDOCell import (
    process_model,
    AIDOCELL_DEFS,
    
)
from etl_utils import (
    compute_attention_from_weights,
    create_adocell_prefix,
    load_results,
    RESULTS_DEFS,
)


In [ ]:
for model_class in [aido_cell_3m, aido_cell_10m, aido_cell_100m]:
    process_model(model_class, OUTPUT_DIR)

In [3]:
weights_dict, gene_annotations, model_metadata = load_results(OUTPUT_DIR, create_adocell_prefix(aido_cell_10m.__name__))

GENES_OF_INTEREST = gene_annotations[RESULTS_DEFS.VOCAB_NAME].sample(10000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in model_metadata[RESULTS_DEFS.ORDERED_VOCABULARY]]

# Compute attention on demand
layer_attn = compute_attention_from_weights(
    weights_dict[RESULTS_DEFS.GENE_EMBEDDING][GENE_MASK,:],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_Q],
    weights_dict[RESULTS_DEFS.ATTENTION_WEIGHTS]['layer_5'][RESULTS_DEFS.W_K]
)

INFO:utils:Loading weights from output/AIDOCell_aido_cell_10m_weights.npz and metadata from output/AIDOCell_aido_cell_10m_metadata.json
INFO:utils:Loading weights from output/AIDOCell_aido_cell_10m_weights.npz
INFO:utils:Loading metadata from output/AIDOCell_aido_cell_10m_metadata.json
INFO:utils:Successfully loaded and validated all results
